# Brain Tumor Detection & Classification

This notebook demonstrates the complete pipeline for brain tumor classification using our hybrid CNN-ViT architecture.

## Features
- Hybrid CNN + Vision Transformer architecture
- Multimodal MRI fusion support
- Self-supervised pre-training
- Radiomics feature integration
- Explainability (Grad-CAM, Attention visualization)

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import yaml

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Load Configuration

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print('Configuration loaded successfully')
print(f"Model: {config['model']['name']}")
print(f"Classes: {config['data']['class_names']}")

## 2. Load and Explore Data

In [ ]:
from src.data import get_data_loaders, BrainTumorDataset

# Update data directory path
DATA_DIR = '../data/raw'  # Update this to your dataset path

# Load data
loaders = get_data_loaders(
    data_dir=DATA_DIR,
    batch_size=16,
    img_size=224,
    num_workers=0  # Set to 0 for notebook
)

print(f"Train samples: {len(loaders['train'].dataset)}")
print(f"Val samples: {len(loaders['val'].dataset)}")
print(f"Test samples: {len(loaders['test'].dataset)}")

In [ ]:
# Visualize sample images
batch = next(iter(loaders['train']))
images = batch['image']
labels = batch['label']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
class_names = config['data']['class_names']

for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i].permute(1, 2, 0).numpy()
        # Denormalize
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = img * std + mean
        img = np.clip(img, 0, 1)
        
        ax.imshow(img)
        ax.set_title(f'{class_names[labels[i]]}')
        ax.axis('off')

plt.suptitle('Sample Training Images', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Create Model

In [ ]:
from src.models import create_model, HybridCNNViT

model = create_model(config)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model size: {total_params * 4 / 1e6:.1f} MB')

In [ ]:
# Test forward pass
with torch.no_grad():
    sample_input = images[:1].to(device)
    output = model(sample_input)
    print(f'Input shape: {sample_input.shape}')
    print(f'Output shape: {output.shape}')
    print(f'Output: {torch.softmax(output, dim=-1)}')

## 4. Training

In [ ]:
from src.training import Trainer

# Reduce epochs for demo
config['training']['epochs'] = 10

trainer = Trainer(model, config, device)

# Train (uncomment to run)
# history = trainer.train(loaders['train'], loaders['val'])

## 5. Evaluation

In [ ]:
from src.evaluation import evaluate_model, get_classification_report
from src.evaluation.visualization import plot_confusion_matrix, plot_roc_curves
from src.evaluation.metrics import compute_roc_curves

# Evaluate on test set
# Uncomment after training
# metrics = evaluate_model(model, loaders['test'], device, class_names)

## 6. Explainability - Grad-CAM

In [ ]:
from src.explainability import GradCAM, apply_gradcam
from src.explainability.gradcam import visualize_gradcam

# Get a sample image
sample = next(iter(loaders['test']))
sample_image = sample['image'][:1].to(device)
sample_label = sample['label'][0].item()

# Denormalize for visualization
original = sample_image[0].cpu().permute(1, 2, 0).numpy()
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
original = np.clip(original * std + mean, 0, 1)

print(f'True label: {class_names[sample_label]}')

# Uncomment after training
# visualize_gradcam(model, sample_image, original, class_names)

## 7. Explainability - Attention Visualization

In [ ]:
from src.explainability.attention_viz import visualize_attention

# Uncomment after training
# visualize_attention(model, sample_image, original, class_names)

## Summary

This notebook demonstrated:
1. Loading and preprocessing brain MRI data
2. Creating the hybrid CNN-ViT model
3. Training with advanced techniques (Mixup, label smoothing)
4. Comprehensive evaluation metrics
5. Model explainability with Grad-CAM and attention visualization

For full training, use:
```bash
python scripts/train.py --config config/config.yaml
```